# Matrices and Transformations

Companion notebook for the [Matrices and Transformations](https://ml-viz-ruby.vercel.app/courses/linear-algebra/02-matrices-and-transformations) lesson.

We'll visualize matrices as geometric transformations and explore matrix multiplication.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Intuition — a matrix is a transformation

A matrix isn't just a grid of numbers — it's a **function that reshapes space**.
Multiplying a vector by a matrix scales, rotates, shears, projects, or flips it, all
at once. Three questions organize everything below: *what does this matrix do to
space?* (read its columns), *what happens if I chain two of them?* (matrix
multiplication = composition), and *did it lose any dimensions?* (the determinant and
rank). This is not abstract: **every layer of a neural network is a matrix multiply**,
so a feel for what matrices do to vectors is a feel for what networks do to data.

## 1. Matrix multiplication, from scratch

Two operations do all the work: **matrix × vector** (transform one point) and
**matrix × matrix** (compose two transforms). Written out, entry `(i, j)` of a product
is the dot product of row `i` with column `j`. The cell implements both with explicit
loops, then demonstrates the single most useful fact about a matrix: **its columns are
exactly where the standard basis vectors land**, so you can read a transform straight
off the numbers.

In [ ]:
def matvec(A, x):                      # A: m×n, x: length n  ->  length m
    return [sum(A[i][j] * x[j] for j in range(len(x))) for i in range(len(A))]

def matmul(A, B):                      # A: m×n, B: n×p  ->  m×p
    n, p = len(B), len(B[0])
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(p)]
            for i in range(len(A))]

A = [[2.0, 1.0],
     [0.0, 3.0]]

# columns of A = images of the basis vectors e1=[1,0], e2=[0,1]
print('A @ e1 =', matvec(A, [1, 0]), ' == first  column of A')
print('A @ e2 =', matvec(A, [0, 1]), ' == second column of A')

# matrix product = "apply B, then A"
B = [[0.0, -1.0],
     [1.0,  0.0]]      # 90° rotation
print('A @ B  =', matmul(A, B))

**What to notice:** `A @ e1` returns `[2, 0]` (A's first column) and `A @ e2` returns
`[1, 3]` (A's second column) — the columns literally *are* the landing spots of the
axes, so any 2×2 matrix is fully described by "where do `[1,0]` and `[0,1]` go?" The
product `A @ B` is a new single matrix that does B's job followed by A's.

## 2. The library way — and the reversibility toolkit

In practice you use the `@` operator and NumPy's linear-algebra routines. The cell
first **checks the from-scratch versions against NumPy** (`@`, and the 2×2
determinant `ad − bc` against `np.linalg.det`), then runs the three functions that
tell you whether a transform is **reversible**: `np.linalg.matrix_rank` (how many
dimensions survive), `np.linalg.det` (zero ⇔ a dimension was lost), and
`np.linalg.inv` (the undo matrix, which only exists when `det ≠ 0`).

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 3.0]])
B = np.array([[0.0, -1.0], [1.0, 0.0]])

# validate the from-scratch implementations
assert np.allclose(A @ B, matmul(A.tolist(), B.tolist()))
assert np.isclose(np.linalg.det(A), A[0,0]*A[1,1] - A[0,1]*A[1,0])   # ad - bc
print('from-scratch matmul & determinant agree with NumPy ✓\n')

# rank / determinant / invertibility for three matrices
matrices = {
    'Full rank (invertible)':    np.array([[3., 1.], [2., 4.]]),
    'Rank deficient (singular)': np.array([[1., 2.], [2., 4.]]),   # row 2 = 2 × row 1
    'Identity':                  np.eye(2),
}
for name, M in matrices.items():
    det = np.linalg.det(M)
    print(f'{name:26s} rank={np.linalg.matrix_rank(M)}  det={det:+.1f}  '
          + ('invertible' if abs(det) > 1e-10 else 'NOT invertible'))

**What to notice:** the assertions pass — the loops and NumPy agree. The full-rank
matrix has `det ≠ 0` and an inverse; the "row 2 = 2 × row 1" matrix is **rank 1** with
`det = 0` and no inverse (it flattened the plane onto a line); the identity has
`det = 1` and changes nothing. Rank, a zero determinant, and non-invertibility are
three views of the same fact.

## 3. Seeing transformations — the unit square

The clearest way to *see* what a matrix does is to watch it act on the **unit square**.
The `draw_transform` helper plots the square (blue), its image `A @ square` (orange),
and the basis vectors before/after. Reading the columns from §1, you can predict each
picture: the next cell runs four classics — a pure scale, a 90° rotation, a projection
onto the x-axis, and a shear.

In [ ]:
def draw_transform(ax, A, title):
    """Draw the unit square and its image under matrix A."""
    # Unit square corners
    square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T   # 2×5
    transformed = A @ square

    ax.plot(*square,  color='#6366f1', lw=2, label='Original', alpha=0.7)
    ax.plot(*transformed, color='#f97316', lw=2, label='Transformed')

    # Basis vectors
    for vec, color in [([1,0], '#2dd4bf'), ([0,1], '#f59e0b')]:
        ax.annotate('', xy=vec, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7))
        transformed_vec = A @ np.array(vec, dtype=float)
        ax.annotate('', xy=transformed_vec, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))

    lim = max(3, abs(transformed).max() + 0.5)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.axhline(0, color='#30344a'); ax.axvline(0, color='#30344a')
    ax.set_title(title, pad=10)
    ax.legend(fontsize=9)

transforms = [
    (np.array([[2,0],[0,1]]), 'Scale x by 2'),
    (np.array([[0,-1],[1,0]]), 'Rotate 90°'),
    (np.array([[1,0],[0,0]]), 'Project onto x-axis'),
    (np.array([[1,1],[0,1]]), 'Shear'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (A, title) in zip(axes, transforms):
    draw_transform(ax, A, title)
plt.suptitle('Linear Transformations of the Unit Square', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**What to notice:** scale widens the square, rotate spins it a quarter turn, shear
slides the top edge while pinning the base. The **projection** is the outlier — it
squashes the square flat onto the x-axis. That collapse from an area to a segment is
the geometric signature of a rank-deficient, non-invertible matrix.

### Composition — chaining transforms

Multiplying matrices chains their transformations. Because a matrix acts on the vector
to its right, `C = R @ S` means **"apply S first, then R"** (read right-to-left). The
cell verifies this by scaling then rotating in two explicit steps and checking it
equals the single combined matrix — and demonstrates that order matters: `R @ S` and
`S @ R` are genuinely different.

In [ ]:
# Scale first, then rotate
scale  = np.array([[2, 0], [0, 0.5]])
rotate = np.array([[0, -1], [1, 0]])    # 90° counterclockwise

# Apply scale THEN rotate: C = R @ S
C = rotate @ scale

x = np.array([1.0, 1.0])
step1 = scale @ x           # after scaling
step2 = rotate @ step1      # after rotation
direct = C @ x              # direct composition

print('Original x:', x)
print('After scale:', step1)
print('After rotate (scale first):', step2)
print('Direct C @ x:', direct)
print('Are they equal?', np.allclose(step2, direct))

# Matrix multiplication is NOT commutative
print('\nR @ S =', C)
print('S @ R =', scale @ rotate)
print('Equal?', np.allclose(C, scale @ rotate))

**What to notice:** `step2` (two moves) equals `direct = C @ x` (one combined matrix)
— which is *why* a whole pipeline can be baked into a single matrix. But `R @ S ≠ S @ R`:
scaling-then-rotating is not rotating-then-scaling. Matrix multiplication is **not
commutative**, and that order is part of the meaning.

## The determinant as area scaling

The determinant of $\begin{bmatrix}a&b\\c&d\end{bmatrix}$ is $ad - bc$. Geometrically it is the **signed factor by which the map scales area**: the unit square becomes a parallelogram of area $|\det|$, with a negative sign when orientation flips. When $\det = 0$ the square collapses to a line — the transformation loses a dimension and is therefore not invertible. Below we verify $|\det|$ equals the image area computed independently with the shoelace formula.

In [ ]:
# The determinant = signed area-scaling factor.
# For a 2x2 matrix [[a,b],[c,d]] the formula is det = ad - bc.
def det2x2(M):
    (a, b), (c, d) = M
    return a * d - b * c

# Shoelace formula: area of the polygon the unit square maps to.
def polygon_area(pts):  # pts: 2 x N (last point need not repeat)
    x, y = pts
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

unit_square = np.array([[0, 1, 1, 0], [0, 0, 1, 1]], dtype=float)  # 4 corners

for name, A in [('scale 2x3', np.array([[2., 0.], [0., 3.]])),
                ('rotate 90', np.array([[0., -1.], [1., 0.]])),
                ('shear',     np.array([[1., 1.], [0., 1.]])),
                ('collapse',  np.array([[1., 2.], [2., 4.]]))]:
    image = A @ unit_square
    manual = det2x2(A)
    print(f'{name:10s}: det(ad-bc)={manual:+.1f}  np.det={np.linalg.det(A):+.1f}  '
          f'image area={polygon_area(image):.1f}  (=|det|={abs(manual):.1f})')
# 'collapse' has det 0 -> the unit square flattens to a line (area 0) -> not invertible.

**What to notice:** on every row the hand-coded `ad − bc` matches NumPy's `det`, and
`|det|` matches the polygon area from the independent shoelace formula — concrete proof
that the determinant *is* the area-scaling factor. The `collapse` matrix has `det = 0`
and image area `0`: the square is flattened to a line and can't be un-flattened.

## Visualizing a matrix transformation — before and after

The side-by-side plot below shows the unit square **before** (left) and **after** (right) a rotation+scaling matrix is applied, making the geometric effect of the transformation concrete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Rotation by 45° composed with scaling (x by 1.5, y by 0.75)
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[1.5, 0.0],
              [0.0, 0.75]])
A = R @ S   # scale first, then rotate

# Unit square corners (close the loop)
square = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]], dtype=float)
transformed = A @ square

# Basis vectors before / after
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])
Ae1, Ae2 = A @ e1, A @ e2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def draw_square_and_basis(ax, sq, b1, b2, title, sq_color='#6366f1',
                          c1='#2dd4bf', c2='#f97316'):
    ax.fill(sq[0], sq[1], color=sq_color, alpha=0.25)
    ax.plot(sq[0], sq[1], color=sq_color, lw=2)
    for vec, col, lbl in [(b1, c1, r'$\mathbf{e}_1$'), (b2, c2, r'$\mathbf{e}_2$')]:
        ax.annotate('', xy=vec, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=col, lw=2.2))
        ax.text(vec[0] + 0.05, vec[1] + 0.05, lbl, color=col, fontsize=12)
    ax.set_xlim(-2.0, 2.0); ax.set_ylim(-2.0, 2.0)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
    ax.axhline(0, color='#30344a', lw=0.8); ax.axvline(0, color='#30344a', lw=0.8)
    ax.set_title(title, pad=12)

# Before
draw_square_and_basis(axes[0], square, e1, e2,
                      'Before: unit square + standard basis')

# After
draw_square_and_basis(axes[1], transformed, Ae1, Ae2,
                      r'After: $A = R_{45°} \cdot S_{1.5 \times 0.75}$',
                      sq_color='#f97316')

# Annotate determinant
det_val = np.linalg.det(A)
axes[1].text(0.02, 0.97,
             f'det(A) = {det_val:.3f}  →  area scales by |det| = {abs(det_val):.3f}',
             transform=axes[1].transAxes, va='top', color='#f59e0b', fontsize=10)

plt.suptitle('Matrix Transformation: Before vs. After', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**What to notice:** `A = R₄₅ · S₁.₅ₓ₀.₇₅` both tilts and restretches the square. Its
determinant is `1.5 × 0.75 = 1.125` — rotation contributes a factor of 1 (it preserves
area) so only the scaling changes the size. The determinant reads off that number
without measuring the picture.

## 4. Limitations & numerical gotchas

- **Order matters.** `A @ B ≠ B @ A` in general — a bug that silently changes results.
- **Singular = no inverse.** When `det = 0` the matrix collapses a dimension and
  `np.linalg.inv` raises. In ML this is *perfect multicollinearity* — redundant
  features breaking the normal equations.
- **Near-singular = ill-conditioned.** A tiny-but-nonzero `det` is worse than useless:
  the **condition number** blows up and the inverse amplifies floating-point error, so
  solutions are numerically garbage. Prefer `np.linalg.solve`/`lstsq` over `inv`.
- **Cost.** Multiplying two `n×n` matrices is `O(n³)`; inversion likewise. This is why
  big models lean on structure (sparsity, low rank) instead of dense inverses.

In [ ]:
# Order matters
R = np.array([[0., -1.], [1., 0.]]); Sx = np.array([[2., 0.], [0., 0.5]])
print('R @ Sx == Sx @ R ?', np.allclose(R @ Sx, Sx @ R))   # False

# Near-singular: small det, huge condition number, unstable inverse
M = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-10]])
print(f'det(M)  = {np.linalg.det(M):.2e}   cond(M) = {np.linalg.cond(M):.2e}')

# Shape mismatch is a hard error
try:
    np.ones((2, 3)) @ np.ones((2, 2))
except ValueError as e:
    print('shape mismatch ->', e)

**What to notice:** `R @ Sx ≠ Sx @ R`; the near-singular `M` has a determinant near
`1e-10` but a condition number around `1e10` — invert it and ten digits of precision
evaporate. And a `(2,3) @ (2,2)` product is a `ValueError`, because the inner
dimensions (3 and 2) must match. These three — order, conditioning, shapes — are the
everyday failure modes when matrix code misbehaves.

## Key takeaways

- A matrix is a **linear transformation**; its **columns are where the basis vectors
  land**, so you can read its effect off the numbers.
- **Matrix multiplication = composition**, applied right-to-left, and is **not
  commutative** — order changes the result.
- The **determinant** is the (signed) area/volume scaling factor; `det = 0` ⇔
  rank-deficient ⇔ a dimension collapsed ⇔ **no inverse**.
- Use `@`, `np.linalg.solve`, `np.linalg.matrix_rank` in practice; avoid explicit
  `inv` on ill-conditioned matrices, and remember matmul is `O(n³)`.

**Next:** [Eigenvalues and Eigenvectors](https://ml-viz-ruby.vercel.app/courses/linear-algebra/03-eigenvalues-and-eigenvectors)
— the special directions a matrix only stretches.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Rotation matrices

A counterclockwise rotation by $\theta$ is the matrix

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Build it, and the checks confirm two facts from the lesson: composing rotations **adds their angles** ($R(45°)R(45°) = R(90°)$), and rotations **preserve area** ($\det R = 1$).

In [ ]:
def rotation(theta_degrees):
    """2x2 matrix that rotates vectors counterclockwise by theta_degrees."""
    t = np.radians(theta_degrees)

    # TODO(you): build [[cos, -sin], [sin, cos]] (hint: np.cos(t), np.sin(t))
    return ...

In [ ]:
# Checks — run me
assert np.allclose(rotation(90) @ [1, 0], [0, 1]), "90° sends the x-axis to the y-axis"
assert np.allclose(rotation(45) @ rotation(45), rotation(90)), "composing rotations adds angles"
assert abs(np.linalg.det(rotation(30)) - 1) < 1e-12, "rotations preserve area (det = 1)"
assert np.allclose(rotation(0), np.eye(2)), "rotating by 0° is the identity"
assert np.allclose(rotation(360), np.eye(2)), "a full turn returns to the identity"
assert np.allclose(rotation(-90) @ rotation(90), np.eye(2)), "rotating by -θ undoes rotating by θ"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rotation(theta_degrees):
    t = np.radians(theta_degrees)
    return np.array([[np.cos(t), -np.sin(t)],
                     [np.sin(t),  np.cos(t)]])
```

</details>

### Exercise 2 — The determinant as area scaling

For a 2×2 matrix $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, the determinant is $ad - bc$, and $|\det A|$ is the factor by which $A$ scales **areas**. Compute it by hand — no `np.linalg.det` allowed.

In [ ]:
def area_scale(A):
    """Factor by which the transformation A scales areas (always >= 0)."""
    A = np.asarray(A, dtype=float)

    # TODO(you): the 2x2 determinant by hand: ad - bc
    det = ...

    # TODO(you): area scaling is its absolute value
    return ...

In [ ]:
# Checks — run me
assert area_scale([[2, 0], [0, 3]]) == 6, "scaling x by 2 and y by 3 scales area by 6"
assert area_scale([[1, 1], [0, 1]]) == 1, "shear slants the square but preserves its area"
assert area_scale([[-1, 0], [0, 1]]) == 1, "reflection flips orientation but keeps area"
assert area_scale([[1, 2], [2, 4]]) == 0, "a singular matrix collapses the plane onto a line"
assert area_scale([[0, 0], [0, 0]]) == 0, "the zero matrix collapses everything to a point"
assert area_scale(np.eye(2)) == 1, "the identity matrix leaves area unchanged"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def area_scale(A):
    A = np.asarray(A, dtype=float)
    det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    return abs(det)
```

</details>

---
## 🎯 Extra practice — matrix operations bank

Eleven small, from-scratch matrix routines from
[DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem) — the kind of
utilities that sit underneath almost every ML library:

- **DML 1** — `matrix-vector-dot-product`
- **DML 2** — `transpose-of-a-matrix`
- **DML 3** — `reshape-matrix`
- **DML 4** — `calculate-mean-by-row-or-column`
- **DML 5** — `scalar-multiplication-of-a-matrix`
- **DML 7** — `matrix-transformation` ($T^{-1}AS$)
- **DML 9** — `matrix-times-matrix`
- **DML 27** — `transformation-matrix-from-basis-b-to-c`
- **DML 35** — `convert-vector-to-diagonal-matrix`
- **DML 55** — `2d-translation-matrix-implementation`
- **DML 119** — `solve-system-of-linear-equations-using-cramer-s-rule`

In [ ]:
def matrix_dot_vector(a, b):
    """DML 1 — matrix @ vector, or -1 if dimensions don't match."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    # TODO(you): if a.shape[1] != len(b): return -1
    if a.shape[1] != len(b):
        return -1
    return (a @ b).tolist()


def transpose_matrix(a):
    """DML 2 — flip rows and columns."""
    # TODO(you): hint — np.array(a).T, then .tolist()
    return ...


def reshape_matrix(a, new_shape):
    """DML 3 — reshape, or [] if the shape is incompatible."""
    a = np.array(a)
    # TODO(you): a.size must equal new_shape[0] * new_shape[1], else return []
    if a.size != new_shape[0] * new_shape[1]:
        return []
    return a.reshape(new_shape).tolist()


def calculate_matrix_mean(matrix, mode):
    """DML 4 — mean along rows ('row') or columns ('column')."""
    matrix = np.array(matrix, dtype=float)
    # TODO(you): axis=1 collapses each ROW to one number, axis=0 collapses each COLUMN
    axis = 1 if mode == 'row' else 0
    return matrix.mean(axis=axis).tolist()


def scalar_multiply(matrix, scalar):
    """DML 5 — multiply every element by a scalar."""
    matrix = np.array(matrix, dtype=float)
    # TODO(you): elementwise scale
    return ...


def matrixmul(a, b):
    """DML 9 — matrix product, or -1 if inner dimensions disagree."""
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    if a.shape[1] != b.shape[0]:
        return -1
    # TODO(you): matrix multiply (hint: @)
    return (...).tolist()


def transform_matrix(A, T, S):
    """DML 7 — T^-1 A S, or -1 if T or S is singular."""
    A, T, S = np.array(A, dtype=float), np.array(T, dtype=float), np.array(S, dtype=float)
    if abs(np.linalg.det(T)) < 1e-10 or abs(np.linalg.det(S)) < 1e-10:
        return -1
    # TODO(you): inv(T) @ A @ S
    return ...


def transform_basis(B, C):
    """DML 27 — transformation matrix P mapping basis B's coordinates to basis C's."""
    B, C = np.array(B, dtype=float), np.array(C, dtype=float)
    # TODO(you): P = C^-1 @ B
    return ...


def make_diagonal(x):
    """DML 35 — turn a 1D vector into a diagonal matrix."""
    x = np.array(x, dtype=float)
    # TODO(you): hint — np.diag
    return ...


def translate_object(points, tx, ty):
    """DML 55 — translate every [x, y] point by (tx, ty)."""
    points = np.array(points, dtype=float)
    # TODO(you): add the offset [tx, ty] to every row
    offset = np.array([tx, ty])
    return (points + offset).tolist()


def cramers_rule(A, b):
    """DML 119 — solve Ax = b via Cramer's rule, or -1 if A is singular."""
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    det_A = np.linalg.det(A)
    if abs(det_A) < 1e-10:
        return -1
    n = len(b)
    x = np.zeros(n)
    for i in range(n):
        Ai = A.copy()
        # TODO(you): replace column i of Ai with b, then x[i] = det(Ai) / det(A)
        Ai[:, i] = b
        x[i] = np.linalg.det(Ai) / det_A
    return x

In [ ]:
# Checks — run me
assert matrix_dot_vector([[1, 2], [2, 4]], [1, 2]) == [5, 10], "DML 1 example"
assert matrix_dot_vector([[1, 2]], [1, 2, 3]) == -1, "mismatched dimensions -> -1"

assert transpose_matrix([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]], "DML 2 example"
assert transpose_matrix([[5]]) == [[5]], "1x1 matrix transposes to itself"

assert reshape_matrix([[1, 2, 3, 4], [5, 6, 7, 8]], (4, 2)) == [[1, 2], [3, 4], [5, 6], [7, 8]], "DML 3 example"
assert reshape_matrix([[1, 2], [3, 4]], (3, 3)) == [], "incompatible shape -> []"

assert calculate_matrix_mean([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 'column') == [4.0, 5.0, 6.0], "DML 4 example"
assert calculate_matrix_mean([[1, 2, 3]], 'row') == [2.0], "single row -> its own mean"

assert scalar_multiply([[1, 2], [3, 4]], 2) == [[2.0, 4.0], [6.0, 8.0]], "DML 5 example"
assert scalar_multiply([[1, 2], [3, 4]], 0) == [[0.0, 0.0], [0.0, 0.0]], "multiplying by 0 zeroes the matrix"

assert matrixmul([[1, 2], [2, 4]], [[2, 1], [3, 4]]) == [[8, 9], [16, 18]], "DML 9 example"
assert matrixmul([[1, 2], [2, 4]], [[2, 1], [3, 4], [4, 5]]) == -1, "shape mismatch -> -1"

assert np.allclose(transform_matrix([[1, 2], [3, 4]], [[2, 0], [0, 2]], [[1, 1], [0, 1]]),
                    [[0.5, 1.5], [1.5, 3.5]]), "DML 7 example"
assert transform_matrix([[1, 2], [3, 4]], [[1, 2], [2, 4]], np.eye(2)) == -1, "singular T -> -1"

P = transform_basis([[1, 0, 0], [0, 1, 0], [0, 0, 1]],
                     [[1, 2.3, 3], [4.4, 25, 6], [7.4, 8, 9]])
assert np.allclose(P, [[-0.6772, -0.0126, 0.2342],
                       [-0.0184, 0.0505, -0.0275],
                       [0.5732, -0.0345, -0.0569]], atol=1e-3), "DML 27 example"

assert np.allclose(make_diagonal([1, 2, 3]), [[1, 0, 0], [0, 2, 0], [0, 0, 3]]), "DML 35 example"
assert np.allclose(make_diagonal([0]), [[0]]), "a single zero -> a 1x1 zero matrix"

assert translate_object([[0, 0], [1, 0], [0.5, 1]], 2, 3) == [[2.0, 3.0], [3.0, 3.0], [2.5, 4.0]], "DML 55 example"
assert translate_object([[0, 0]], 0, 0) == [[0.0, 0.0]], "translating by (0,0) is a no-op"

x = cramers_rule([[2, -1, 3], [4, 2, 1], [-6, 1, -2]], [5, 10, -3])
assert np.allclose(x, [0.1667, 3.3333, 2.6667], atol=1e-3), "DML 119 example"
assert cramers_rule([[1, 2], [2, 4]], [1, 2]) == -1, "singular A (dependent rows) -> -1"

print("✅ Extra-practice matrix operations bank passed")

<details>
<summary>💡 Show solution</summary>

```python
def matrix_dot_vector(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    if a.shape[1] != len(b):
        return -1
    return (a @ b).tolist()


def transpose_matrix(a):
    return np.array(a).T.tolist()


def reshape_matrix(a, new_shape):
    a = np.array(a)
    if a.size != new_shape[0] * new_shape[1]:
        return []
    return a.reshape(new_shape).tolist()


def calculate_matrix_mean(matrix, mode):
    matrix = np.array(matrix, dtype=float)
    axis = 1 if mode == 'row' else 0
    return matrix.mean(axis=axis).tolist()


def scalar_multiply(matrix, scalar):
    matrix = np.array(matrix, dtype=float)
    return (matrix * scalar).tolist()


def matrixmul(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    if a.shape[1] != b.shape[0]:
        return -1
    return (a @ b).tolist()


def transform_matrix(A, T, S):
    A, T, S = np.array(A, dtype=float), np.array(T, dtype=float), np.array(S, dtype=float)
    if abs(np.linalg.det(T)) < 1e-10 or abs(np.linalg.det(S)) < 1e-10:
        return -1
    return np.linalg.inv(T) @ A @ S


def transform_basis(B, C):
    B, C = np.array(B, dtype=float), np.array(C, dtype=float)
    return np.linalg.inv(C) @ B


def make_diagonal(x):
    return np.diag(np.array(x, dtype=float))


def translate_object(points, tx, ty):
    points = np.array(points, dtype=float)
    return (points + np.array([tx, ty])).tolist()


def cramers_rule(A, b):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    det_A = np.linalg.det(A)
    if abs(det_A) < 1e-10:
        return -1
    n = len(b)
    x = np.zeros(n)
    for i in range(n):
        Ai = A.copy()
        Ai[:, i] = b
        x[i] = np.linalg.det(Ai) / det_A
    return x
```

</details>

### Extra practice — solving linear systems bank

Six ways to solve $Ax = b$, from direct elimination to iterative methods used
when $A$ is too large or sparse to invert directly:

- **DML 11** — `solve-linear-equations-using-jacobi-method`
- **DML 48** — `implement-reduced-row-echelon-form-rref-function`
- **DML 57** — `gauss-seidel-method-for-solving-linear-systems`
- **DML 58** — `gaussian-elimination-for-solving-linear-systems`
- **DML 63** — `implement-the-conjugate-gradient-method-for-solving-linear-systems`
- **DML 68** — `find-the-image-of-a-matrix-using-row-echelon-form`

In [ ]:
def solve_jacobi(A, b, n):
    """DML 11 — Jacobi iteration for Ax = b, n sweeps, rounded to 4 decimals."""
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    m = len(b)
    x = np.zeros(m)
    for _ in range(n):
        x_new = np.zeros(m)
        for i in range(m):
            # TODO(you): sum A[i,j]*x[j] over every j != i
            s = sum(A[i, j] * x[j] for j in range(m) if j != i)
            x_new[i] = (b[i] - s) / A[i, i]
        x = x_new
    return [round(float(v), 4) for v in x]


def gauss_seidel(A, b, n, x_ini=None):
    """DML 57 — like Jacobi, but each update uses the FRESHEST values immediately."""
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    m = len(b)
    x = np.zeros(m) if x_ini is None else np.array(x_ini, dtype=float)
    for _ in range(n):
        for i in range(m):
            # TODO(you): same formula as Jacobi, but x is updated IN PLACE
            s = sum(A[i, j] * x[j] for j in range(m) if j != i)
            x[i] = (b[i] - s) / A[i, i]
    return x


def gaussian_elimination(A, b):
    """DML 58 — forward elimination with partial pivoting + back-substitution."""
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    n = len(b)
    aug = np.hstack([A, b.reshape(-1, 1)])
    for i in range(n):
        pivot = np.argmax(np.abs(aug[i:, i])) + i
        aug[[i, pivot]] = aug[[pivot, i]]
        for j in range(i + 1, n):
            # TODO(you): eliminate column i from row j using row i
            factor = aug[j, i] / aug[i, i]
            aug[j, i:] -= factor * aug[i, i:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        # TODO(you): back-substitute using already-solved x[i+1:]
        x[i] = (aug[i, -1] - aug[i, i + 1:n] @ x[i + 1:n]) / aug[i, i]
    return x


def conjugate_gradient(A, b, n, x0=None, tol=1e-8):
    """DML 63 — CG for symmetric positive-definite A."""
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.array(x0, dtype=float)
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    for _ in range(n):
        Ap = A @ p
        # TODO(you): step size alpha = rs_old / (p . Ap)
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if np.sqrt(rs_new) < tol:
            break
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x


def rref(matrix):
    """DML 48 — reduced row echelon form via Gauss-Jordan elimination."""
    M = np.array(matrix, dtype=float)
    rows, cols = M.shape
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        pivot = np.argmax(np.abs(M[r:, c])) + r
        if abs(M[pivot, c]) < 1e-10:
            continue
        M[[r, pivot]] = M[[pivot, r]]
        # TODO(you): normalize the pivot row so the leading entry becomes 1
        M[r] = M[r] / M[r, c]
        for i in range(rows):
            if i != r:
                M[i] = M[i] - M[i, c] * M[r]
        r += 1
    return M


def matrix_image(A):
    """DML 68 — basis for the column space: the original pivot columns of A."""
    A = np.array(A, dtype=float)
    M = A.copy()
    rows, cols = M.shape
    r = 0
    pivot_cols = []
    for c in range(cols):
        if r >= rows:
            break
        pivot = np.argmax(np.abs(M[r:, c])) + r
        if abs(M[pivot, c]) < 1e-10:
            continue
        M[[r, pivot]] = M[[pivot, r]]
        M[r] = M[r] / M[r, c]
        for i in range(rows):
            if i != r:
                M[i] = M[i] - M[i, c] * M[r]
        # TODO(you): column c of the ORIGINAL matrix A is a pivot column — keep it
        pivot_cols.append(c)
        r += 1
    return A[:, pivot_cols]

In [ ]:
# Checks — run me
A_j = [[5, -2, 3], [-3, 9, 1], [2, -1, -7]]
b_j = [-1, 2, 3]
assert solve_jacobi(A_j, b_j, 2) == [0.146, 0.2032, -0.5175], "DML 11 example"

A_gs = np.array([[4, 1, 2], [3, 5, 1], [1, 1, 3]], dtype=float)
b_gs = np.array([4, 7, 3], dtype=float)
assert np.allclose(gauss_seidel(A_gs, b_gs, 100), [0.5, 1.0, 0.5], atol=1e-3), \
    "converges to the exact solution of the system"

A_ge = np.array([[2, 8, 4], [2, 5, 1], [4, 10, -1]], dtype=float)
b_ge = np.array([2, 5, 1], dtype=float)
assert np.allclose(gaussian_elimination(A_ge, b_ge), [11.0, -4.0, 3.0]), "DML 58 example"
assert np.allclose(gaussian_elimination(np.eye(3), np.array([1., 2., 3.])), [1, 2, 3]), \
    "solving with the identity just returns b"

A_cg = np.array([[4, 1], [1, 3]], dtype=float)
b_cg = np.array([1, 2], dtype=float)
assert np.allclose(conjugate_gradient(A_cg, b_cg, 5), [0.09090909, 0.63636364], atol=1e-6), "DML 63 example"
assert np.allclose(conjugate_gradient(np.eye(2), np.array([5., 7.]), 5), [5, 7]), \
    "the identity system converges in one step"

M = rref(np.array([[1, 2, -1, -4], [2, 3, -1, -11], [-2, 0, -3, 22]], dtype=float))
assert np.allclose(M, [[1, 0, 0, -8], [0, 1, 0, 1], [0, 0, 1, -2]], atol=1e-8), "DML 48 example"
zero_row_matrix = rref(np.array([[1, 2], [2, 4]], dtype=float))
assert np.allclose(zero_row_matrix[1], [0, 0]), "a dependent row reduces to all zeros"

img = matrix_image(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float))
assert np.allclose(img, [[1, 2], [4, 5], [7, 8]]), "DML 68 example"
full_rank = matrix_image(np.eye(3))
assert full_rank.shape == (3, 3), "a full-rank matrix keeps every column"

print("✅ Extra-practice linear systems bank passed")

<details>
<summary>💡 Show solution</summary>

```python
def solve_jacobi(A, b, n):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    m = len(b)
    x = np.zeros(m)
    for _ in range(n):
        x_new = np.zeros(m)
        for i in range(m):
            s = sum(A[i, j] * x[j] for j in range(m) if j != i)
            x_new[i] = (b[i] - s) / A[i, i]
        x = x_new
    return [round(float(v), 4) for v in x]


def gauss_seidel(A, b, n, x_ini=None):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    m = len(b)
    x = np.zeros(m) if x_ini is None else np.array(x_ini, dtype=float)
    for _ in range(n):
        for i in range(m):
            s = sum(A[i, j] * x[j] for j in range(m) if j != i)
            x[i] = (b[i] - s) / A[i, i]
    return x


def gaussian_elimination(A, b):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    n = len(b)
    aug = np.hstack([A, b.reshape(-1, 1)])
    for i in range(n):
        pivot = np.argmax(np.abs(aug[i:, i])) + i
        aug[[i, pivot]] = aug[[pivot, i]]
        for j in range(i + 1, n):
            factor = aug[j, i] / aug[i, i]
            aug[j, i:] -= factor * aug[i, i:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (aug[i, -1] - aug[i, i + 1:n] @ x[i + 1:n]) / aug[i, i]
    return x


def conjugate_gradient(A, b, n, x0=None, tol=1e-8):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.array(x0, dtype=float)
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    for _ in range(n):
        Ap = A @ p
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if np.sqrt(rs_new) < tol:
            break
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x


def rref(matrix):
    M = np.array(matrix, dtype=float)
    rows, cols = M.shape
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        pivot = np.argmax(np.abs(M[r:, c])) + r
        if abs(M[pivot, c]) < 1e-10:
            continue
        M[[r, pivot]] = M[[pivot, r]]
        M[r] = M[r] / M[r, c]
        for i in range(rows):
            if i != r:
                M[i] = M[i] - M[i, c] * M[r]
        r += 1
    return M


def matrix_image(A):
    A = np.array(A, dtype=float)
    M = A.copy()
    rows, cols = M.shape
    r = 0
    pivot_cols = []
    for c in range(cols):
        if r >= rows:
            break
        pivot = np.argmax(np.abs(M[r:, c])) + r
        if abs(M[pivot, c]) < 1e-10:
            continue
        M[[r, pivot]] = M[[pivot, r]]
        M[r] = M[r] / M[r, c]
        for i in range(rows):
            if i != r:
                M[i] = M[i] - M[i, c] * M[r]
        pivot_cols.append(c)
        r += 1
    return A[:, pivot_cols]
```

</details>